# 6. Ablation, baselines and robustness

What does each field contribute, do the persistence diagrams matter, and how stable is the result?

In [ ]:
import sys, pickle
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "analysis"), str(ROOT / "src"), str(ROOT)]
import numpy as np
import matplotlib.pyplot as plt
import figlib as L
L.style()
PAPER = ROOT / "results" / "epuck" / "paper"
RES = ROOT / "results" / "epuck"
load = lambda p: pickle.load(open(p, "rb"))

## Field ablation

Eleven field combinations, each fused with the same rule (equal weight per moment present), over the ten datasets
and the 99 settings: 990 values per combination.

In [ ]:
import make_figures as M
from IPython.display import Image, display
if (PAPER / "ablation.pkl").exists():
    M.fig_ablation(); display(Image(filename=str(ROOT / "figures/fig_ablation.png"), width=950))
else:
    print("ablation still running: python run_sweep.py ablation")

## Baselines under the same clustering

Every representation goes through the same rank affinity, the same diffusion and the same AP over the same 99 settings,
and its taxonomy is built on its own plateau; only the representation changes.

* **Order parameters**: polarisation, milling, speed, spread (mean and standard deviation over the run).
* **Topology of positions (CROCKER)**: Betti-0/1 curves of the robot positions at 60 instants.
* **Field statistics**: the same seeds and fields, value distributions instead of persistence diagrams.

In [ ]:
import subprocess, sys as _s
subprocess.run([_s.executable, str(ROOT / "analysis/make_numbers.py")], capture_output=True)
tab = ROOT / "manuscript/tables/baselines.tex"
print(tab.read_text() if tab.exists() else "baselines still running: python analysis/baselines_eval.py")

## Robustness

**Rotation.** Five runs, one per mission, were rotated and processed again; the table gives the distance between a run and
its rotated copy relative to the median distance between two runs of the same mission (median, max in parentheses).

In [ ]:
print((ROOT / "manuscript/tables/invariance.tex").read_text())

**Convergence of affinity propagation.** With damping 0.9 the messages change slowly, so the algorithm needs enough
iterations to settle. We keep the window of 100 unchanged iterations and the ceiling of 1000 iterations of the original
implementation. `analysis/ap_check.py` runs every setting of Top-1 and of Top-10, the largest dataset, again with a
longer window, a higher ceiling, another damping and another tie-breaking noise, which must give the same partitions if
the settings let the algorithm finish, and with the shorter window of scikit-learn (15 iterations):

In [ ]:
AC = load(PAPER / "ap_check.pkl")
for top in (1, 10):
    ref = AC[(top, "paper")]; it = np.array(ref["iterations"]); n = len(it)
    print(f"Top-{top}, settings of the paper: converged at {ref['converged']}/{n} settings, "
          f"iterations median {np.median(it):.0f}, max {it.max()}")
    for (t_, name), r in AC.items():
        if t_ == top and name != "paper":
            print(f"   {name:12s}: identical partitions {r['settings_identical']}/{n} (smallest ARI {r['settings_ari_min']:.3f}), "
                  f"consensus ARI {r['consensus_ari']:.3f}, iterations max {r['iterations_max']}")

**Fusion architectures and standard SNF** (appendix of the paper): stacking diffusion stages, or using the heat kernel of
standard SNF instead of ranks, makes the number of clusters drift or collapse.

In [ ]:
for name in ("designs", "standard_snf"):
    p = ROOT / f"manuscript/tables/{name}.tex"
    print(p.read_text() if p.exists() else f"{name}: not computed yet")